## Introductie 

Deze les behandelt: 
- Wat functieaanroepen zijn en hun gebruikssituaties 
- Hoe een functieaanroep te maken met Azure OpenAI 
- Hoe een functieaanroep te integreren in een applicatie 

## Leerdoelen 

Na het voltooien van deze les weet je hoe en begrijp je: 

- Het doel van het gebruik van functieaanroepen 
- Het opzetten van functieaanroepen met de Azure Open AI-service 
- Het ontwerpen van effectieve functieaanroepen voor het gebruiksscenario van je applicatie 


## Functieaanroepen begrijpen 

Voor deze les willen we een functie bouwen voor onze educatieve startup die gebruikers in staat stelt een chatbot te gebruiken om technische cursussen te vinden. We zullen cursussen aanbevelen die passen bij hun vaardigheidsniveau, huidige rol en technologie van interesse. 

Om dit te voltooien zullen we een combinatie gebruiken van: 
 - `Azure Open AI` om een chatervaring voor de gebruiker te creëren
 - `Microsoft Learn Catalog API` om gebruikers te helpen cursussen te vinden op basis van het verzoek van de gebruiker 
 - `Function Calling` om de vraag van de gebruiker te nemen en door te sturen naar een functie die het API-verzoek uitvoert. 

Om te beginnen, laten we kijken waarom we functieaanroepen in de eerste plaats zouden willen gebruiken: 


### Waarom Functie-aanroepen  

Als je al een andere les in deze cursus hebt afgerond, begrijp je waarschijnlijk de kracht van het gebruik van grote taalmodellen (LLM's). Hopelijk zie je ook enkele van hun beperkingen.  

Functie-aanroepen is een functie van de Azure Open AI Service om de volgende beperkingen te overwinnen:  
1) Consistent antwoordformaat  
2) Mogelijkheid om gegevens van andere bronnen van een applicatie te gebruiken in een chatcontext  

Voor functie-aanroepen waren de antwoorden van een LLM ongestructureerd en inconsistent. Ontwikkelaars moesten complexe validatiecode schrijven om ervoor te zorgen dat ze met elke variatie van een antwoord konden omgaan.  

Gebruikers konden geen antwoorden krijgen op vragen als "Wat is het huidige weer in Stockholm?". Dit kwam omdat modellen beperkt waren tot de tijd waarop de data was getraind.  

Laten we eens kijken naar het onderstaande voorbeeld dat dit probleem illustreert:  

Stel dat we een database met studentgegevens willen maken zodat we de juiste cursus aan hen kunnen voorstellen. Hieronder hebben we twee beschrijvingen van studenten die erg vergelijkbaar zijn in de gegevens die ze bevatten.  


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


We willen dit naar een LLM sturen om de gegevens te parseren. Dit kan later in onze applicatie worden gebruikt om dit naar een API te sturen of op te slaan in een database.

Laten we twee identieke prompts maken waarin we de LLM instrueren over welke informatie wij geïnteresseerd zijn:


We willen dit naar een LLM sturen om de onderdelen te analyseren die belangrijk zijn voor ons product. Zodat we twee identieke prompts kunnen maken om de LLM te instrueren: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Na het maken van deze twee prompts sturen we ze naar de LLM door gebruik te maken van `client.responses.create`. We slaan de prompt op in de variabele `input` en wijzen de rol toe aan `user`. Dit is om een bericht van een gebruiker te simuleren dat naar een chatbot wordt geschreven. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Nu kunnen we beide verzoeken naar de LLM sturen en de reactie die we ontvangen onderzoeken. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Ook al zijn de prompts hetzelfde en lijken de beschrijvingen op elkaar, we kunnen verschillende formaten van de `Grades` eigenschap krijgen.

Als je de bovenstaande cel meerdere keren uitvoert, kan het formaat `3.7` of `3.7 GPA` zijn.

Dit komt omdat het LLM ongestructureerde data in de vorm van de geschreven prompt ontvangt en ook ongestructureerde data retourneert. We moeten een gestructureerd formaat hebben zodat we weten wat we kunnen verwachten bij het opslaan of gebruiken van deze data.

Door functioneel aanroepen te gebruiken, kunnen we ervoor zorgen dat we gestructureerde data terugkrijgen. Bij gebruik van functioneel aanroepen voert het LLM eigenlijk geen functies uit. In plaats daarvan creëren we een structuur voor het LLM om te volgen bij zijn antwoorden. We gebruiken die gestructureerde antwoorden dan om te weten welke functie we in onze applicaties moeten aanroepen.
 


![Diagram van Functie Aansturoverzicht](../../../../translated_images/nl/Function-Flow.083875364af4f4bb.webp)


We kunnen dan nemen wat er wordt teruggegeven door de functie en dit terugsturen naar de LLM. De LLM zal vervolgens reageren met natuurlijke taal om de vraag van de gebruiker te beantwoorden. 


### Gebruikscasussen voor het gebruiken van functie-aanroepen 

**Externe hulpmiddelen aanroepen** 
Chatbots zijn uitstekend in het geven van antwoorden op vragen van gebruikers. Door het gebruik van functie-aanroepen kunnen de chatbots gebruikersberichten gebruiken om bepaalde taken uit te voeren. Bijvoorbeeld, een student kan de chatbot vragen om "Een e-mail te sturen naar mijn docent met de mededeling dat ik meer hulp nodig heb bij dit onderwerp". Dit kan een functie-aanroep doen naar `send_email(to: string, body: string)`


**API- of database-queries maken**
Gebruikers kunnen informatie vinden door middel van natuurlijke taal die wordt omgezet in een geformatteerde query of API-verzoek. Een voorbeeld hiervan kan een docent zijn die vraagt "Wie zijn de studenten die de laatste opdracht hebben afgerond" wat een functie-aanroep kan zijn naar `get_completed(student_name: string, assignment: int, current_status: string)`


**Gestructureerde gegevens maken**
Gebruikers kunnen een tekstblok of CSV gebruiken en het LLM gebruiken om belangrijke informatie eruit te halen. Bijvoorbeeld, een student kan een Wikipedia-artikel over vredesakkoorden omzetten om AI-flashcards te maken. Dit kan gedaan worden met een functie genaamd `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Je eerste functieaanroep maken 

Het proces van het maken van een functieaanroep omvat 3 hoofdtrappen: 
1. Het aanroepen van de Chat Completions API met een lijst van je functies en een gebruikersbericht 
2. Lees de reactie van het model om een actie uit te voeren, bijvoorbeeld een functie of API-aanroep uitvoeren 
3. Maak nog een aanroep naar de Chat Completions API met de reactie van je functie om die informatie te gebruiken om een antwoord aan de gebruiker te maken. 


![Stroom van een Functieaanroep](../../../../translated_images/nl/LLM-Flow.3285ed8caf4796d7.webp)


### Elementen van een functieverzoek 

#### Invoer door gebruikers 

De eerste stap is het creëren van een gebruikersbericht. Dit kan dynamisch worden toegewezen door de waarde van een tekstinvoer te nemen, of je kunt hier een waarde toewijzen. Als dit de eerste keer is dat je met de Chat Completions API werkt, moeten we de `role` en de `content` van het bericht definiëren. 

De `role` kan `system` zijn (regels maken), `assistant` (het model) of `user` (de eindgebruiker). Voor functieverzoeken zullen we dit toewijzen als `user` en een voorbeeldvraag stellen. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Functies maken. 

Vervolgens definiëren we een functie en de parameters van die functie. We gebruiken hier slechts één functie genaamd `search_courses` maar je kunt meerdere functies maken.

**Belangrijk** : Functies zijn opgenomen in het systeembericht aan de LLM en zullen meetellen voor het aantal beschikbare tokens dat je hebt. 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Definities** 

`name` - De naam van de functie die we willen laten aanroepen. 

`description` - Dit is de beschrijving van hoe de functie werkt. Hier is het belangrijk om specifiek en duidelijk te zijn 

`parameters` - Een lijst van waarden en het formaat dat je wilt dat het model produceert in zijn antwoord 


`type` - Het gegevenstype waarin de eigenschappen zullen worden opgeslagen 

`properties` - Lijst van de specifieke waarden die het model zal gebruiken voor zijn antwoord 


`name` - de naam van de eigenschap die het model zal gebruiken in zijn geformatteerde antwoord 

`type` - Het gegevenstype van deze eigenschap 

`description` - Beschrijving van de specifieke eigenschap 


**Optioneel**

`required` - vereiste eigenschap voor het voltooien van de functiemaakroep 


### Het aanroepen van de functie 
Na het definiëren van een functie moeten we deze nu opnemen in het aanroepen van de Chat Completion API. Dit doen we door `functions` toe te voegen aan het verzoek. In dit geval `functions=functions`. 

Er is ook een optie om `function_call` in te stellen op `auto`. Dit betekent dat we de LLM laten beslissen welke functie moet worden aangeroepen op basis van het gebruikersbericht in plaats van die zelf toe te wijzen.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Laten we nu naar de reactie kijken en zien hoe deze is geformatteerd: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Je kunt zien dat de naam van de functie wordt aangeroepen en uit het bericht van de gebruiker kon het LLM de gegevens vinden die overeenkomen met de argumenten van de functie. 


## 3.Functieaanroepen integreren in een applicatie. 


Nadat we het geformatteerde antwoord van de LLM hebben getest, kunnen we dit nu integreren in een applicatie. 

### De flow beheren 

Om dit in onze applicatie te integreren, laten we de volgende stappen nemen: 

Eerst maken we de oproep naar de Open AI-services en slaan het bericht op in een variabele genaamd `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Nu definiëren we de functie die de Microsoft Learn API zal aanroepen om een lijst met cursussen te verkrijgen: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Als beste praktijk zullen we vervolgens controleren of het model een functie wil aanroepen. Daarna maken we een van de beschikbare functies aan en koppelen deze aan de functie die wordt aangeroepen. 
Vervolgens nemen we de argumenten van de functie en mappen deze naar argumenten van de LLM.

Ten slotte voegen we het functieverzoekbericht toe en de waarden die door het `search_courses`-bericht zijn teruggegeven. Dit geeft de LLM alle informatie die het nodig heeft om
te reageren op de gebruiker met natuurlijke taal. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Nu zullen we het bijgewerkte bericht naar de LLM sturen zodat we een reactie in natuurlijke taal kunnen ontvangen in plaats van een API-JSON-geformatteerde reactie. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Code Uitdaging 

Geweldig werk! Om je leren over Azure Open AI Function Calling voort te zetten, kun je bouwen: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - Meer parameters van de functie die kunnen helpen bij het vinden van meer cursussen voor leerlingen. Je kunt hier de beschikbare API-parameters vinden: 
 - Maak een andere functie-aanroep die meer informatie van de leerling vraagt, zoals hun moedertaal 
 - Maak foutafhandeling wanneer de functie-aanroep en/of API-aanroep geen geschikte cursussen teruggeeft 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
